# Baseline Results with Pixtral 12B

In [2]:
import pandas as pd

# Load the data
merged_data = pd.read_csv('../data/preprocess/merged_data.csv')
receipt_data = pd.read_csv('../data/preprocess/receipt_data.csv')
# Display the first few rows of the dataframe
display(merged_data)
display(receipt_data.head())

,Bon_ID,Artikelnummer,Artikel,Stückartikel,Anzahl_Kellen,Gewicht_Kelle,Gewicht_Teller,kcal_Teller,kj_Teller,Fett_Teller,ges_Fettsäuren_Teller,Kohlenhydrate_Teller,Zucker_Teller,Eiweiß_Teller,Salz_Teller,Menge_Rückläufer,Prozent_Rückläufer
0,480654,91862294,Kartoffelpüree,N,2,"60,0","120,0","129,960","544,440","5,640","0,600","16,440","0,480","2,040","0,960",58.0,48.33
1,480654,91511750,Erbsen,N,1,"40,0","40,0","36,400","151,600","1,160","0,120","3,480","1,200","2,000","0,040",12.0,30.00
2,480654,91511769,Möhre,N,1,"40,0","40,0","18,800","78,800","1,000","0,080","2,240","2,160","0,240","0,280",14.0,35.00
3,480654,91551949,Bratenjus,N,1,"50,0","50,0","42,000","177,500","0,350","0,050","5,500","3,400","4,000","0,700",26.0,52.00
4,480654,91533800,Hähnchen,J,1,"91,0","91,0","90,090","380,380","2,093","0,455","1,820","0,637","16,107","1,274",50.0,54.95
5,480656,91511793,Kartoffeln,N,1,"95,0","95,0","56,050","231,800","0,000","0,000","11,685","0,475","1,425","0,760",62.0,65.26
6,480656,91511750,Erbsen,N,1,"40,0","40,0","36,400","151,600","1,160","0,120","3,480","1,200","2,000","0,040",24.0,60.00
7,480656,91511769,Möhre,N,1,"40,0","40,0","18,800","78,800","1,000","0,080","2,240","2,160","0,240","0,280",38.0,95.00
8,480656,91511823,Braune Sauce,N,2,"35,0","70,0","82,600","344,400","4,830","0,350","7,630","5,880","1,260","1,680",18.0,25.71
9,480656,91533800,Hähnchen,J,1,"91,0","91,0","90,090","380,380","2,093","0,455","1,820","0,637","16,107","1,274",42.0,46.15


,Bon_ID,Gericht,Portionsgröße,Gewicht_vorher,kcal_vorher,kj_vorher,Fett_vorher,ges_Fettsäuren_vorher,Kohlenhydrate_vorher,Zucker_vorher,...,Salz_vorher,Gewicht_nachher,kcal_nachher,kj_nachher,Fett_nachher,ges_fettsäuren_nachher,Kohlenhydrate_nachher,Zucker_nachher,Eiweiß_nachher,Salz_nachher
0,480654,"Hähnchenbrustfilet mit Erbsen und Möhren, Kart...",Normale Portion,341,"315,250","1.323,220","10,293","1,305","28,830","7,477",...,"3,404",197,"182,124","764,441","5,946","0,754","16,655","4,320","14,204","1,967"
1,480656,"Hähnchen mit Kartoffeln , Erbsen Möhre und Bra...",Normale Portion,336,"234,940","983,980","4,603","0,725","25,385","8,252",...,"3,824",166,"116,072","486,133","2,274","0,358","12,541","4,077","10,356","1,889"
2,480657,Rostbratwurst mit Sauerkraut und Kartoffeln an...,Normale Portion,325,"326,570","1.363,620","19,630","6,490","23,745","8,215",...,"5,430",195,"195,942","818,172","11,778","3,894","14,247","4,929","7,113","3,258"
3,480658,"Rostbratwurst mit Rotkohl , Kartoffelpüree , B...",Normale Portion,300,"360,980","1.512,760","21,220","6,540","28,000","8,570",...,"4,830",214,"257,499","1.079,102","15,137","4,665","19,973","6,113","8,503","3,445"
4,480659,Schweineschnitzel nach Jägerart mit Pilz-Rahms...,Normale Portion,365,"307,590","1.279,400","9,260","3,510","33,125","8,755",...,"3,830",237,"199,723","830,734","6,013","2,279","21,509","5,685","13,327","2,487"


### Using Pixtral Hosted on the HPI Assistant

 - Using [DSPY](https://dspy-docs.vercel.app/) for automatic Prompt Engineering

In [3]:
import dspy
from dotenv import load_dotenv
import os
load_dotenv()
api_key = os.getenv('API_KEY')

lm = dspy.LM("openai/mistralai/Pixtral-12B-2409",
                 api_base="https://chat.hpi-sci.de/api/", 
                 api_key=api_key)
dspy.configure(lm=lm)

/Users/felixboelter/Documents/GitHub/food-waste/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:

def get_ingredients(bon_id : int) -> str: 
    """
    This function retrieves the ingredients and their corresponding weights for a given Bon ID.
    It calculates the total weight for each ingredient and the overall total weight.

    Args:
    bon_id (int): The Bon ID for which to retrieve the ingredients.

    Returns:
    str: A description of the ingredients with their total weights and the overall total weight.
    """
    artikel = merged_data[merged_data["Bon_ID"] == bon_id]['Artikel'].to_list() 
    gewicht_kelle = merged_data[merged_data["Bon_ID"] == bon_id]['Gewicht_Kelle'].to_list() 
    anzahl_kelle = merged_data[merged_data["Bon_ID"] == bon_id]['Anzahl_Kellen'].to_list() 
    gewicht_kelle = [float(value.replace(",", ".")) for value in gewicht_kelle]
    total_gewicht = sum([anzahl_kelle[i] * gewicht_kelle[i] for i in range(len(gewicht_kelle))])
    description = f""
    for art, gewicht, anzahl in zip(artikel, gewicht_kelle, anzahl_kelle):
        gesamt_gewicht = gewicht * anzahl
        description += f"{art} - Gesamt Gewicht: {gesamt_gewicht}\n"
    description += "Total Gewicht: " + str(total_gewicht)
    return description

In [5]:
class LeftOverPlate(dspy.Signature):
    """
    DSPY Signature for Left Over Plate analysis.

    This signature is used to analyze the contents of a left-over plate. It includes:
    - An image of the left-over plate.
    - The original ingredients and their corresponding weights.
    - The ingredients visible on the left-over plate.
    - A detailed description of each ingredient's state and location on the plate.
    - An estimated weight of each left-over ingredient.
    - The total summed weight of all left-over ingredients.
    """
    
    image_1: dspy.Image = dspy.InputField(
        desc="An image of the left-over plate. This image is used to visually analyze the remaining food items."
    )
    
    original_ingredients: str = dspy.InputField(
        desc="A list of the ingredients and their corresponding weights on the original plate before consumption."
    )
    
    ingredients: str = dspy.OutputField(
        desc="A list of ingredients that are visible on the left-over plate after consumption."
    )
    
    ingredient_description: dict[str, list[str, str]] = dspy.OutputField(
        desc="For each ingredient, provide a detailed description including: "
             "the amount visible on the left-over plate, whether they are large pieces, tiny/small pieces, "
             "mixed with something else, and their location on the plate."
    )
    
    ingredient_weight: dict[str, float] = dspy.OutputField(
        desc="An estimated weight of each left-over ingredient. Very small pieces/remnants should have a low weight (5-10 grams). Left-over sauce is usually 1-3 grams."
    )
    
    json_out: float = dspy.OutputField(
        desc="The total summed weight of all left-over ingredients. This is the final output representing the total weight of the leftovers."
    )


In [9]:
import glob
import os
import json
# Get all image files in the directory
image_files = glob.glob('../data/images_new/*.jpg')
output_list = []
# Loop over each image file
predict = dspy.ChainOfThought(LeftOverPlate)

for image_file in image_files:
    # Extract the name before the first dot
    image_basename = os.path.basename(image_file).split('.')[0]
    image_name = os.path.basename(image_file)
    # Call get_ingredients with the extracted number
    ingredients = get_ingredients(int(image_basename))
    # image_url = f"https://huggingface.co/datasets/AI-ServicesBB/food-waste/resolve/main/images/{image_name}"

    ### TODO
    ## Use dpsy.Image.from_PIL to load the image
    # image = dpsy.Image.from_PIL(PIL.Image.open(...))
    ### TODO
    image = dspy.Image.from_file(image_file)
    try:
        out = predict(image_1=image, original_ingredients=ingredients)
        print(out.json_out)
    except Exception as e:
        print(f"Error with image {image_name}")
        print(e)
        continue
    # Add Output the a file
    output = {}
    output["type"] = out.ingredient_weight
    output["total_weight"] = out.json_out
    output["image"] = image_name
    output_list.append(output)
   
with open('data_3.json', 'w') as file:
    file.write(json.dumps(output_list, indent=4))

77.0
117.0
82.0
160.0
100.0
70.0
160.0
145.0
114.0
100.0
109.0
108.0
130.0
115.0
80.0
100.0
120.0
77.0
137.0
109.0
137.0
122.0
140.0
95.0
80.0
61.0
30.0
87.0
104.0
132.0
137.0
72.0
30.0
72.0
103.0
104.0
125.0
142.0
62.0
70.0
75.0
37.0
90.0
72.0
77.0
72.0
50.0
104.0
65.0
92.0
45.0
102.0
62.0
100.0
80.0
40.0
110.0
70.0
100.0
100.0
100.0
62.0
100.0
62.0
100.0
65.0
61.0
67.0
100.0
72.0
62.0
90.0
72.0
122.0
61.0
45.0
102.0
102.0
62.0
127.0
70.0
95.0
65.0
45.0
137.0
104.0
110.0
122.0
55.0
41.0
90.0
82.0
120.0
102.0
77.0
112.0
62.0
72.0
100.0
120.0
80.0
100.0
72.0
101.0
60.0
61.0
80.0
72.0
100.0
90.0
110.0
100.0
130.0
125.0
120.0
61.0
61.0
120.0
102.0


In [10]:
data_df = pd.read_json('data_3.json')
display(data_df.head())
df = pd.DataFrame()
# Extract the image basename from the image column
df['image_basename'] = data_df['image'].apply(lambda x: x.split('.')[0])
df['total_weight'] = data_df['total_weight']

# Calculate the average total weight for each image basename
average_weights = df.groupby('image_basename')['total_weight'].mean().reset_index()

# Merge the average weights back into the original dataframe
df = df.merge(average_weights, on='image_basename', suffixes=('', '_average'))
df.drop_duplicates(subset='image_basename', inplace=True)
df['image_basename'] = df['image_basename'].astype(str)
receipt_data['Bon_ID'] = receipt_data['Bon_ID'].astype(str)
receipt_data['ground_truth'] = receipt_data['Gewicht_vorher'] - receipt_data['Gewicht_nachher']
# Merge Gewicht_nachher from receipt_data into df
df = df.merge(receipt_data[['Bon_ID', 'ground_truth']], left_on='image_basename', right_on='Bon_ID', how='left')

# Rename Gewicht_nachher to Ground truth gewicht
df.rename(columns={'ground_truth': 'Ground truth weight'}, inplace=True)

# Drop the Bon_ID column as it is no longer needed
df.drop(columns=['Bon_ID'], inplace=True)
df.drop(columns=['total_weight'], inplace=True)
# Display the first few rows of the updated dataframe
display(df)

,type,total_weight,image
0,"{'Kartoffeln': 20.0, 'Erbsen': 10.0, 'Möhre': ...",77,480659.00015.jpg
1,"{'Kartoffeln': 30.0, 'Erbsen': 10.0, 'Möhre': ...",117,480659.00001.jpg
2,"{'Kartoffelpüree': 40.0, 'Erbsen': 10.0, 'Möhr...",82,480654.00016.jpg
3,"{'Kartoffelpüree': 60.0, 'Erbsen': 20.0, 'Möhr...",160,480654.00002.jpg
4,"{'Kartoffeln': 30.0, 'Sauerkraut': 40.0, 'Brau...",100,480657.00022.jpg


,image_basename,total_weight_average,Ground truth weight
0,480659,110.789474,128
1,480654,101.444444,144
2,480657,90.826087,130
3,480658,71.388889,86
4,480660,70.260870,130
5,480656,105.388889,170
